# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 11. Green Street Historical POI Integration, 2019-2025

This notebook converts the confidential Green Street tenant-premises history into annual MSOA indicators and integrates them into H1a, H1b and H3.

Green Street has confirmed the tenant/premises status distinction, the consumer-facing vacancy rule, the High Street denominator, the consistency of vacancy methodology across 2019-2025 and the meanings of `CreatedDate` and `ClosedDate`. The dates record when Green Street entered the observed opening or closure into its database rather than the unknown real-world event date. Because locations are normally revisited on a six-month cycle, event timing is interval-observed with up to approximately six months of uncertainty. Annual aggregation is therefore the main temporal specification.

The notebook never exports addresses or row-level tenant records. Outputs are restricted MSOA-year aggregates.

In [ ]:
from pathlib import Path
import os
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 160)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
OUT_DIR = BASE / "outputs" / "restricted_greenstreet_historical_poi_integration"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "historical_poi": BASE / "Greater London Export (2019 to 2026).csv",
    "current_poi": BASE / "Greater_London_POI_Churn_2026-07-01-1110.csv",
    "msoa": BASE / "outputs" / "restricted_msoa_origin_exposure_analysis" / "london_msoa_2021_boundaries.geojson",
    "workplace": BASE / "outputs" / "restricted_h1_fine_grained_analysis" / "h1_destination_all_submarket_msoa_targets.csv",
    "origin_exposure": BASE / "outputs" / "restricted_source_disagreement_h1_refinement" / "h1_origin_msoa_absolute_and_normalised_exposure.csv",
}
missing = [name for name, path in FILES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing inputs: {missing}")

PALETTE = {
    "orange": "#d88958", "orange_dark": "#a65432", "blue": "#5f9fc7",
    "blue_dark": "#2d6f9f", "green": "#6f9f8f", "red": "#b45f55",
    "grey": "#8d9497", "light": "#f2f1ed", "ink": "#263238",
}
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9})

## 1. Confirmed Operational Definitions

In [ ]:
CONFIG = {
    "analysis_years": list(range(2019, 2026)),
    "snapshot_month": 12,
    "snapshot_day": 31,
    "actual_business_types": ["Independent", "Multiple"],
    "excluded_actual_categories": ["Non-Retail", "Miscellaneous"],
    "vacancy_subcategory": "Vacant Properties",
    "long_term_vacancy_days": 365 * 3,
    "annual_state_rule": "latest active record within each PremiseId",
    "event_date_interpretation": "Green Street database detection date, not exact real-world event date (confirmed)",
    "event_time_uncertainty": "up to approximately six months under the normal survey cycle",
    "temporal_aggregation": "calendar-year cuts; half-year reporting is possible but not used as the main specification",
    "tenant_status_definition": "Live=current trading occupier; Closed=former occupier; Vacant=unit linked to a vacant-property record (confirmed)",
    "premise_status_definition": "current building/premises state, distinct from a specific tenant's status (confirmed)",
    "vacancy_universe": "live retail businesses plus vacant units; Non-Retail, Miscellaneous and demolished premises excluded (confirmed)",
    "vacancy_observation_rule": "vacant until a consumer can enter and purchase; under-offer units remain vacant (confirmed)",
}
display(pd.DataFrame([{"rule": key, "setting": value} for key, value in CONFIG.items()]))

## 2. Read, Standardise and Categorise the Historical Records

In [ ]:
history = pd.read_csv(FILES["historical_poi"], low_memory=False)
history.columns = [column.strip() for column in history.columns]
for column in ["Tenant ID", "PremiseId"]:
    history[column] = history[column].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
history["CreatedDate"] = pd.to_datetime(history["CreatedDate"], errors="coerce")
history["ClosedDate"] = pd.to_datetime(history["ClosedDate"], errors="coerce")
history["Latitude"] = pd.to_numeric(history["Latitude"], errors="coerce")
history["Longitude"] = pd.to_numeric(history["Longitude"], errors="coerce")

current = pd.read_csv(FILES["current_poi"], usecols=["SUBCATEGORY", "CATEGORY"], low_memory=False)
category_crosswalk = (
    current.dropna(subset=["SUBCATEGORY", "CATEGORY"])
    .groupby("SUBCATEGORY")["CATEGORY"]
    .agg(category=lambda values: values.mode().iloc[0], category_variants="nunique")
    .reset_index()
    .rename(columns={"SUBCATEGORY": "SubCategory"})
)
history = history.merge(category_crosswalk, on="SubCategory", how="left")
history["record_type"] = np.select(
    [
        history["BusinessType"].isin(CONFIG["actual_business_types"]),
        history["SubCategory"].eq(CONFIG["vacancy_subcategory"]),
        history["SubCategory"].eq("Demolished Properties"),
        history["SubCategory"].eq("Merged Properties"),
        history["SubCategory"].eq("Split Properties"),
    ],
    ["Actual business", "Vacant", "Demolished", "Merged", "Split"],
    default="Other dummy",
)
history["is_retail_business"] = history["record_type"].eq("Actual business") & ~history["category"].isin(CONFIG["excluded_actual_categories"])

quality = pd.DataFrame(
    {
        "metric": [
            "Rows", "Unique tenants", "Unique premises", "Retail-business records",
            "Vacancy records", "Missing coordinates", "Closed before created",
            "Live status with ClosedDate", "Closed status without ClosedDate",
        ],
        "value": [
            len(history), history["Tenant ID"].nunique(), history["PremiseId"].nunique(),
            int(history["is_retail_business"].sum()), int(history["record_type"].eq("Vacant").sum()),
            int((history["Latitude"].isna() | history["Longitude"].isna()).sum()),
            int((history["ClosedDate"].notna() & (history["ClosedDate"] < history["CreatedDate"])).sum()),
            int((history["Status"].eq("Live") & history["ClosedDate"].notna()).sum()),
            int((history["Status"].eq("Closed") & history["ClosedDate"].isna()).sum()),
        ],
    }
)
quality.to_csv(OUT_DIR / "historical_poi_quality_summary.csv", index=False)
display(quality)

## 3. Assign Premises to 2021 MSOAs

In [ ]:
msoa = gpd.read_file(FILES["msoa"]).to_crs("EPSG:27700")
premise_locations = (
    history.dropna(subset=["PremiseId", "Latitude", "Longitude"])
    .sort_values(["PremiseId", "CreatedDate"])
    .drop_duplicates("PremiseId", keep="last")[["PremiseId", "Latitude", "Longitude"]]
)
premise_points = gpd.GeoDataFrame(
    premise_locations,
    geometry=gpd.points_from_xy(premise_locations["Longitude"], premise_locations["Latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:27700")
premise_msoa = gpd.sjoin(
    premise_points, msoa[["MSOA21CD", "MSOA21NM", "geometry"]], how="left", predicate="within"
).drop(columns="index_right", errors="ignore")
history = history.merge(premise_msoa[["PremiseId", "MSOA21CD", "MSOA21NM"]], on="PremiseId", how="left")
print("Records assigned to MSOA:", history["MSOA21CD"].notna().sum(), "/", len(history))
print("Premises assigned to MSOA:", premise_msoa["MSOA21CD"].notna().sum(), "/", len(premise_msoa))

## 4. Build Annual MSOA States and Events

A record is active at a snapshot when `CreatedDate <= snapshot` and `ClosedDate > snapshot` or is missing. When several records are active for one premise, the most recently created record determines the annual state. The number of such overlaps is retained as a quality indicator.

Recorded openings and closures use actual retail-business records only. They represent Green Street detection dates rather than exact real-world event dates. Vacancy duration is calculated only for the latest active `Vacant Properties` state and is interpreted as recorded vacancy duration, with up to approximately six months of onset uncertainty.

In [ ]:
state_rows = []
category_rows = []
retail = history[history["is_retail_business"] & history["MSOA21CD"].notna()].copy()

for year in CONFIG["analysis_years"]:
    year_start = pd.Timestamp(year=year, month=1, day=1)
    snapshot = pd.Timestamp(year=year, month=CONFIG["snapshot_month"], day=CONFIG["snapshot_day"])
    active_mask = history["CreatedDate"].le(snapshot) & (history["ClosedDate"].isna() | history["ClosedDate"].gt(snapshot))
    candidates = history.loc[active_mask & history["MSOA21CD"].notna()].sort_values(
        ["PremiseId", "CreatedDate", "Tenant ID"]
    )
    active_count = candidates.groupby("PremiseId").size().rename("active_record_count")
    latest = candidates.drop_duplicates("PremiseId", keep="last").merge(active_count, on="PremiseId", how="left")
    latest["vacancy_duration_days"] = np.where(
        latest["record_type"].eq("Vacant"), (snapshot - latest["CreatedDate"]).dt.days, np.nan
    )
    latest["is_long_term_vacant"] = latest["record_type"].eq("Vacant") & latest["vacancy_duration_days"].ge(CONFIG["long_term_vacancy_days"])
    latest["is_active_retail"] = latest["is_retail_business"]
    latest["is_active_vacant"] = latest["record_type"].eq("Vacant")
    latest["is_chain"] = latest["is_active_retail"] & latest["BusinessType"].eq("Multiple")

    opened = retail[retail["CreatedDate"].between(year_start, snapshot, inclusive="both")]
    closed = retail[retail["ClosedDate"].between(year_start, snapshot, inclusive="both")]
    opening_counts = opened.groupby(["MSOA21CD", "MSOA21NM"]).agg(
        opening_events=("Tenant ID", "size"), opening_premises=("PremiseId", "nunique")
    )
    closure_counts = closed.groupby(["MSOA21CD", "MSOA21NM"]).agg(
        closure_events=("Tenant ID", "size"), closure_premises=("PremiseId", "nunique")
    )
    event_by_premise = pd.concat(
        [
            opened.groupby(["MSOA21CD", "MSOA21NM", "PremiseId"]).size().rename("open_events"),
            closed.groupby(["MSOA21CD", "MSOA21NM", "PremiseId"]).size().rename("close_events"),
        ],
        axis=1,
    ).fillna(0)
    event_by_premise["replacement_events"] = event_by_premise[["open_events", "close_events"]].min(axis=1)
    replacements = event_by_premise.groupby(["MSOA21CD", "MSOA21NM"]).agg(
        replacement_events=("replacement_events", "sum"),
        replacement_premises=("replacement_events", lambda values: int(values.gt(0).sum())),
    )

    active_retail = latest[latest["is_active_retail"]].copy()
    stock = latest.groupby(["MSOA21CD", "MSOA21NM"]).agg(
        active_retail_premises=("is_active_retail", "sum"),
        active_vacant_premises=("is_active_vacant", "sum"),
        long_term_vacant_premises=("is_long_term_vacant", "sum"),
        chain_retail_premises=("is_chain", "sum"),
        overlapping_active_premises=("active_record_count", lambda values: int(values.gt(1).sum())),
    )
    diversity = active_retail.groupby(["MSOA21CD", "MSOA21NM"])["category"].apply(
        lambda values: -np.sum((values.value_counts(normalize=True) * np.log(values.value_counts(normalize=True))).values)
    ).rename("category_shannon")
    category_count = active_retail.groupby(["MSOA21CD", "MSOA21NM"])["category"].nunique().rename("active_categories")

    annual = pd.concat(
        [stock, diversity, category_count, opening_counts, closure_counts, replacements], axis=1
    ).reset_index()
    count_columns = [
        "active_retail_premises", "active_vacant_premises", "long_term_vacant_premises",
        "chain_retail_premises", "overlapping_active_premises", "active_categories",
        "opening_events", "opening_premises", "closure_events", "closure_premises",
        "replacement_events", "replacement_premises",
    ]
    annual[count_columns] = annual[count_columns].fillna(0)
    annual["year"] = year
    universe = annual["active_retail_premises"] + annual["active_vacant_premises"]
    annual["vacancy_share"] = annual["active_vacant_premises"] / universe.replace(0, np.nan)
    annual["long_term_vacancy_share"] = annual["long_term_vacant_premises"] / universe.replace(0, np.nan)
    annual["long_term_share_of_vacancy"] = annual["long_term_vacant_premises"] / annual["active_vacant_premises"].replace(0, np.nan)
    annual["opening_rate"] = annual["opening_events"] / annual["active_retail_premises"].replace(0, np.nan)
    annual["closure_rate"] = annual["closure_events"] / annual["active_retail_premises"].replace(0, np.nan)
    annual["turnover_rate"] = (annual["opening_events"] + annual["closure_events"]) / (2 * annual["active_retail_premises"].replace(0, np.nan))
    annual["replacement_rate"] = annual["replacement_events"] / annual["active_retail_premises"].replace(0, np.nan)
    annual["net_formation_rate"] = (annual["opening_events"] - annual["closure_events"]) / annual["active_retail_premises"].replace(0, np.nan)
    annual["chain_share"] = annual["chain_retail_premises"] / annual["active_retail_premises"].replace(0, np.nan)
    state_rows.append(annual)

    category_events = pd.concat(
        [
            opened.groupby(["MSOA21CD", "MSOA21NM", "category"]).size().rename("openings"),
            closed.groupby(["MSOA21CD", "MSOA21NM", "category"]).size().rename("closures"),
        ],
        axis=1,
    ).fillna(0).reset_index()
    category_events["year"] = year
    category_events["net_change"] = category_events["openings"] - category_events["closures"]
    category_rows.append(category_events)

msoa_year = pd.concat(state_rows, ignore_index=True)
msoa_category_year = pd.concat(category_rows, ignore_index=True)
msoa_year.to_csv(OUT_DIR / "greenstreet_historical_poi_msoa_year.csv", index=False)
msoa_category_year.to_csv(OUT_DIR / "greenstreet_historical_poi_msoa_category_year.csv", index=False)
print("MSOA-year observations:", len(msoa_year), "MSOAs:", msoa_year["MSOA21CD"].nunique())
display(msoa_year.groupby("year")[["active_retail_premises", "active_vacant_premises", "opening_events", "closure_events"]].sum())

## 5. Build 2019-Baseline Change Indicators

In [ ]:
baseline_metrics = [
    "active_retail_premises", "vacancy_share", "long_term_vacancy_share",
    "opening_rate", "closure_rate", "turnover_rate", "replacement_rate",
    "net_formation_rate", "chain_share", "category_shannon",
]
baseline = msoa_year[msoa_year["year"].eq(2019)][["MSOA21CD"] + baseline_metrics].rename(
    columns={metric: f"{metric}_2019" for metric in baseline_metrics}
)
panel = msoa_year[msoa_year["year"].isin([2023, 2024, 2025])].merge(baseline, on="MSOA21CD", how="inner")
panel["active_retail_log_change_2019"] = np.log1p(panel["active_retail_premises"]) - np.log1p(panel["active_retail_premises_2019"])
panel["log_active_retail_2019"] = np.log1p(panel["active_retail_premises_2019"])
for metric in baseline_metrics[1:]:
    panel[f"{metric}_change_2019"] = panel[metric] - panel[f"{metric}_2019"]
panel.to_csv(OUT_DIR / "greenstreet_historical_poi_msoa_post_panel.csv", index=False)

indicator_dictionary = pd.DataFrame([
    ["active_retail_premises", "Latest recorded active Independent/Multiple retail tenant per premise, excluding Non-Retail and Miscellaneous", "H1, H3", "Confirmed; observation timing may lag by up to about six months"],
    ["opening_rate", "Recorded retail CreatedDate events / active retail premises", "H1, H3", "Confirmed detection-date measure; not exact opening date"],
    ["closure_rate", "Recorded retail ClosedDate events / active retail premises", "H1, H3", "Confirmed detection-date measure; not exact closure date"],
    ["turnover_rate", "(Recorded openings + recorded closures) / (2 x active retail premises)", "H3", "Confirmed annual detection-based measure"],
    ["replacement_rate", "Matched recorded opening and closure events within premise-year / stock", "H3", "Confirmed annual detection-based measure"],
    ["vacancy_share", "Latest recorded active Vacant Properties / eligible live retail plus vacant premises", "H1, H3", "Vacancy universe confirmed"],
    ["long_term_vacancy_share", "Recorded active vacancy spell aged 3+ years / retail plus vacant premises", "H3", "Confirmed; vacancy onset has up to about six months uncertainty"],
    ["chain_share", "Multiple-name active retail premises / active retail premises", "H3", "Usable"],
    ["category_shannon", "Shannon diversity of active retail categories", "H1, H3", "Usable after category crosswalk"],
], columns=["indicator", "definition", "hypothesis_role", "status"])
indicator_dictionary.to_csv(OUT_DIR / "greenstreet_historical_indicator_dictionary.csv", index=False)
display(indicator_dictionary)

## 6. London-Wide Data Trajectories

In [ ]:
london_year = msoa_year.groupby("year", as_index=False).agg(
    active_retail_premises=("active_retail_premises", "sum"),
    active_vacant_premises=("active_vacant_premises", "sum"),
    openings=("opening_events", "sum"),
    closures=("closure_events", "sum"),
    replacements=("replacement_events", "sum"),
)
london_year["vacancy_share"] = london_year["active_vacant_premises"] / (
    london_year["active_retail_premises"] + london_year["active_vacant_premises"]
)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5))
axes[0].plot(london_year["year"], london_year["active_retail_premises"], marker="o", color=PALETTE["blue_dark"], label="Active retail")
axes[0].plot(london_year["year"], london_year["active_vacant_premises"], marker="o", color=PALETTE["orange_dark"], label="Vacant")
axes[0].set_title("Reconstructed premises stock")
axes[0].legend(frameon=False)
axes[1].plot(london_year["year"], london_year["openings"], marker="o", color=PALETTE["green"], label="Openings")
axes[1].plot(london_year["year"], london_year["closures"], marker="o", color=PALETTE["red"], label="Closures")
axes[1].set_title("Retail tenant events")
axes[1].legend(frameon=False)
axes[2].plot(london_year["year"], london_year["vacancy_share"], marker="o", color=PALETTE["orange_dark"])
axes[2].set_title("Reconstructed vacancy share")
axes[2].set_ylabel("Share")
for ax in axes:
    ax.set_xlabel("Year")
    ax.grid(alpha=0.2)
fig.suptitle("Green Street Historical POI: Annual Recorded-State Reconstruction", x=0.02, ha="left", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.90])
fig.savefig(FIG_DIR / "fig_01_greenstreet_historical_annual_reconstruction.png", bbox_inches="tight")
plt.show()

## 7. H1a Workplace-Destination Models

In [ ]:
workplace = pd.read_csv(FILES["workplace"])
workplace_panel = panel.merge(workplace, on=["MSOA21CD", "MSOA21NM"], how="inner")

h1a_specs = {
    "Active retail stock": ("active_retail_log_change_2019", "log_active_retail_2019"),
    "Vacancy share": ("vacancy_share_change_2019", "vacancy_share_2019"),
    "Opening rate": ("opening_rate_change_2019", "opening_rate_2019"),
    "Closure rate": ("closure_rate_change_2019", "closure_rate_2019"),
    "Net formation": ("net_formation_rate_change_2019", "net_formation_rate_2019"),
    "Turnover rate": ("turnover_rate_change_2019", "turnover_rate_2019"),
}

def fit_h1a(outcome, baseline_column):
    columns = ["MSOA21CD", "year", "study_submarket", "workplace_msoa_station_shock", outcome, baseline_column]
    clean = workplace_panel[columns].replace([np.inf, -np.inf], np.nan).dropna().copy()
    clean["exposure"] = (clean["workplace_msoa_station_shock"] - clean["workplace_msoa_station_shock"].mean()) / clean["workplace_msoa_station_shock"].std(ddof=0)
    year_dummies = pd.get_dummies(clean["year"].astype(str), prefix="year", drop_first=True, dtype=float)
    market_dummies = pd.get_dummies(clean["study_submarket"].astype(str), prefix="market", drop_first=True, dtype=float)
    X = pd.concat([clean[["exposure", baseline_column]].astype(float).reset_index(drop=True), year_dummies.reset_index(drop=True), market_dummies.reset_index(drop=True)], axis=1)
    X = sm.add_constant(X)
    fit = sm.OLS(clean[outcome].astype(float).reset_index(drop=True), X).fit(
        cov_type="cluster", cov_kwds={"groups": clean["MSOA21CD"].reset_index(drop=True)}
    )
    return {"n_obs": int(fit.nobs), "n_msoas": clean["MSOA21CD"].nunique(), "beta": fit.params["exposure"], "se": fit.bse["exposure"], "p_value": fit.pvalues["exposure"], "r_squared": fit.rsquared}

h1a_rows = []
for label, (outcome, baseline_column) in h1a_specs.items():
    result = fit_h1a(outcome, baseline_column)
    result["outcome"] = label
    h1a_rows.append(result)
h1a_results = pd.DataFrame(h1a_rows)
h1a_results["ci_low"] = h1a_results["beta"] - 1.96 * h1a_results["se"]
h1a_results["ci_high"] = h1a_results["beta"] + 1.96 * h1a_results["se"]
h1a_results.to_csv(OUT_DIR / "h1a_greenstreet_historical_poi_models.csv", index=False)

fig, ax = plt.subplots(figsize=(8.6, 4.2))
y = np.arange(len(h1a_results))
ax.errorbar(h1a_results["beta"], y, xerr=1.96 * h1a_results["se"], fmt="o", color=PALETTE["orange_dark"], capsize=3)
ax.axvline(0, color=PALETTE["grey"], linewidth=0.8)
ax.set_yticks(y, h1a_results["outcome"])
ax.set_xlabel("Effect of a 1-SD workplace commuter shock (95% CI)")
ax.set_title("H1a: Green Street Historical Retail Outcomes", loc="left")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_02_h1a_greenstreet_historical_poi_coefficients.png", bbox_inches="tight")
plt.show()
display(h1a_results.round(4))

## 8. H1b Residential-Origin Models

In [ ]:
origin = pd.read_csv(FILES["origin_exposure"])
origin_panel = panel.merge(origin, left_on="MSOA21CD", right_on="origin_msoa", how="inner")
h1b_specs = {
    "Active retail stock": ("active_retail_log_change_2019", "log_active_retail_2019"),
    "Opening rate": ("opening_rate_change_2019", "opening_rate_2019"),
    "Net formation": ("net_formation_rate_change_2019", "net_formation_rate_2019"),
    "Vacancy share": ("vacancy_share_change_2019", "vacancy_share_2019"),
    "Turnover rate": ("turnover_rate_change_2019", "turnover_rate_2019"),
}
exposure_specs = {
    "Absolute exposure": "absolute_shock_exposure_per_1000",
    "Normalised exposure": "normalised_shock_exposure_per_1000_outbound",
}

def fit_h1b(outcome, baseline_column, exposure_column):
    columns = ["MSOA21CD", "year", "origin_lad_name_guess", "flow_weighted_commute_distance_km", exposure_column, outcome, baseline_column]
    clean = origin_panel[columns].replace([np.inf, -np.inf], np.nan).dropna().copy()
    clean["exposure"] = (clean[exposure_column] - clean[exposure_column].mean()) / clean[exposure_column].std(ddof=0)
    clean["distance"] = (clean["flow_weighted_commute_distance_km"] - clean["flow_weighted_commute_distance_km"].mean()) / clean["flow_weighted_commute_distance_km"].std(ddof=0)
    year_dummies = pd.get_dummies(clean["year"].astype(str), prefix="year", drop_first=True, dtype=float)
    X = pd.concat([clean[["exposure", "distance", baseline_column]].astype(float).reset_index(drop=True), year_dummies.reset_index(drop=True)], axis=1)
    X = sm.add_constant(X)
    fit = sm.OLS(clean[outcome].astype(float).reset_index(drop=True), X).fit(
        cov_type="cluster", cov_kwds={"groups": clean["MSOA21CD"].reset_index(drop=True)}
    )
    return {"n_obs": int(fit.nobs), "n_msoas": clean["MSOA21CD"].nunique(), "n_lads": clean["origin_lad_name_guess"].nunique(), "beta": fit.params["exposure"], "se": fit.bse["exposure"], "p_value": fit.pvalues["exposure"], "r_squared": fit.rsquared}

h1b_rows = []
for outcome_label, (outcome, baseline_column) in h1b_specs.items():
    for exposure_label, exposure_column in exposure_specs.items():
        result = fit_h1b(outcome, baseline_column, exposure_column)
        result.update({"outcome": outcome_label, "exposure": exposure_label})
        h1b_rows.append(result)
h1b_results = pd.DataFrame(h1b_rows)
h1b_results.to_csv(OUT_DIR / "h1b_greenstreet_historical_poi_models.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.5), gridspec_kw={"width_ratios": [0.85, 1.45]})
offsets = {"Absolute exposure": -0.12, "Normalised exposure": 0.12}
colors = {"Absolute exposure": PALETTE["blue_dark"], "Normalised exposure": PALETTE["orange_dark"]}
outcome_groups = {
    "A. Retail stock": ["Active retail stock"],
    "B. Retail rates": ["Opening rate", "Net formation", "Vacancy share", "Turnover rate"],
}
for ax, (panel_title, outcomes) in zip(axes, outcome_groups.items()):
    for exposure_label in exposure_specs:
        subset = h1b_results[h1b_results["exposure"].eq(exposure_label)].set_index("outcome").reindex(outcomes)
        y = np.arange(len(subset)) + offsets[exposure_label]
        ax.errorbar(
            subset["beta"], y, xerr=1.96 * subset["se"], fmt="o",
            color=colors[exposure_label], capsize=3, label=exposure_label,
        )
    ax.axvline(0, color=PALETTE["grey"], linewidth=0.8)
    ax.set_yticks(np.arange(len(outcomes)), outcomes)
    ax.set_title(panel_title, loc="left", fontsize=11)
    ax.grid(axis="x", alpha=0.2)
axes[0].set_xlabel("Effect on log stock change")
axes[1].set_xlabel("Effect on rate change")
axes[1].legend(frameon=False, loc="best")
fig.suptitle("H1b: Green Street Historical Retail Outcomes", x=0.06, ha="left", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_03_h1b_greenstreet_historical_poi_coefficients.png", bbox_inches="tight")
plt.show()
display(h1b_results.round(4))

## 9. H3 Turnover, Vacancy and Stagnation Pathways

In [ ]:
h3_columns = [
    "turnover_rate_change_2019", "replacement_rate_change_2019", "vacancy_share_change_2019",
    "long_term_vacancy_share_change_2019", "active_retail_log_change_2019", "chain_share_change_2019",
]
h3 = workplace_panel.groupby(["MSOA21CD", "MSOA21NM"], as_index=False)[h3_columns].mean().merge(
    workplace[["MSOA21CD", "workplace_msoa_station_shock", "study_submarket"]], on="MSOA21CD", how="left"
)
turnover_cut = h3["turnover_rate_change_2019"].median()
stagnation_cut = h3["long_term_vacancy_share_change_2019"].median()
h3["pathway"] = np.select(
    [
        h3["turnover_rate_change_2019"].ge(turnover_cut) & h3["long_term_vacancy_share_change_2019"].lt(stagnation_cut),
        h3["turnover_rate_change_2019"].ge(turnover_cut) & h3["long_term_vacancy_share_change_2019"].ge(stagnation_cut),
        h3["turnover_rate_change_2019"].lt(turnover_cut) & h3["long_term_vacancy_share_change_2019"].ge(stagnation_cut),
    ],
    ["Adaptive/high churn", "Turbulent stress", "Low-churn vacancy"],
    default="Stable/low churn",
)
h3.to_csv(OUT_DIR / "h3_greenstreet_adjustment_pathways.csv", index=False)

h3_specs = {
    "Active retail stock": ("active_retail_log_change_2019", "log_active_retail_2019"),
    "Vacancy share": ("vacancy_share_change_2019", "vacancy_share_2019"),
    "Long-term vacancy": ("long_term_vacancy_share_change_2019", "long_term_vacancy_share_2019"),
    "Turnover rate": ("turnover_rate_change_2019", "turnover_rate_2019"),
    "Replacement rate": ("replacement_rate_change_2019", "replacement_rate_2019"),
    "Chain share": ("chain_share_change_2019", "chain_share_2019"),
}
h3_model_rows = []
for outcome_label, (outcome, baseline_column) in h3_specs.items():
    result = fit_h1a(outcome, baseline_column)
    result["outcome"] = outcome_label
    h3_model_rows.append(result)
h3_models = pd.DataFrame(h3_model_rows)
h3_models["ci_low"] = h3_models["beta"] - 1.96 * h3_models["se"]
h3_models["ci_high"] = h3_models["beta"] + 1.96 * h3_models["se"]
h3_models.to_csv(OUT_DIR / "h3_greenstreet_continuous_models.csv", index=False)

fig, ax = plt.subplots(figsize=(8.0, 5.4))
scatter = ax.scatter(
    h3["turnover_rate_change_2019"], h3["long_term_vacancy_share_change_2019"],
    c=h3["workplace_msoa_station_shock"], cmap="Oranges", s=45, edgecolor="white", linewidth=0.4,
)
ax.axvline(turnover_cut, color=PALETTE["grey"], linestyle="--", linewidth=0.8)
ax.axhline(stagnation_cut, color=PALETTE["grey"], linestyle="--", linewidth=0.8)
ax.set_xlabel("Change in recorded tenant turnover rate: 2023-2025 mean vs 2019")
ax.set_ylabel("Change in recorded long-term vacancy share: 2023-2025 mean vs 2019")
ax.set_title("H3 Exploratory Adjustment Pathways in Workplace MSOAs", loc="left")
x_pad = max((h3["turnover_rate_change_2019"].max() - h3["turnover_rate_change_2019"].min()) * 0.04, 0.001)
y_pad = max((h3["long_term_vacancy_share_change_2019"].max() - h3["long_term_vacancy_share_change_2019"].min()) * 0.04, 0.001)
ax.text(turnover_cut + x_pad, stagnation_cut + y_pad, "Turbulent stress", fontsize=9, color=PALETTE["grey"])
ax.text(turnover_cut + x_pad, stagnation_cut - y_pad, "Adaptive / high churn", fontsize=9, color=PALETTE["grey"], va="top")
ax.text(turnover_cut - x_pad, stagnation_cut + y_pad, "Low-churn vacancy", fontsize=9, color=PALETTE["grey"], ha="right")
ax.text(turnover_cut - x_pad, stagnation_cut - y_pad, "Stable / low churn", fontsize=9, color=PALETTE["grey"], ha="right", va="top")
cbar = fig.colorbar(scatter, ax=ax, shrink=0.78)
cbar.set_label("Workplace commuter-shock score")
ax.grid(alpha=0.15)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_04_h3_turnover_vacancy_pathways.png", bbox_inches="tight")
plt.show()
display(h3["pathway"].value_counts())

fig, ax = plt.subplots(figsize=(8.6, 4.4))
y = np.arange(len(h3_models))
ax.errorbar(h3_models["beta"], y, xerr=1.96 * h3_models["se"], fmt="o", color=PALETTE["orange_dark"], capsize=3)
ax.axvline(0, color=PALETTE["grey"], linewidth=0.8)
ax.set_yticks(y, h3_models["outcome"])
ax.set_xlabel("Effect of a 1-SD workplace commuter shock (95% CI)")
ax.set_title("H3: Continuous Turnover, Vacancy and Consolidation Tests", loc="left")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_05_h3_greenstreet_continuous_coefficients.png", bbox_inches="tight")
plt.show()
display(h3_models.round(4))

## 10. Final Interpretation Rules

The tenant/premises status distinction, consumer-facing vacancy rule, High Street denominator, 2019-2025 methodological consistency and event-date interpretation are confirmed. `CreatedDate` and `ClosedDate` are Green Street database detection dates, not exact real-world opening and closure dates. Recorded events may therefore lag the underlying change by up to approximately six months under the normal survey cycle.

Reporting rules:

- use terms such as **recorded opening**, **recorded closure** and **recorded vacancy duration**;
- aggregate events annually in the main analysis, consistent with the observation cycle and the dissertation's 2019 versus 2023-2025 design;
- do not interpret individual dates as exact business opening or closure dates;
- treat a three-year long-term vacancy threshold as approximate at the boundary because the recorded onset may lag by up to about six months;
- retain simultaneous active tenant records as a reported quality flag and use the latest recorded state per `PremiseId` for the annual reconstruction;
- use the confirmed eligible retail-plus-vacant denominator, excluding Non-Retail, Miscellaneous and demolished records.

The H3 pathway labels are exploratory, median-based summaries. They are not final clusters or causal classifications.